# Validacao ampliada NSOS / Jamba / OxtaMem - T4

Notebook exportado apos execucao real em Tesla T4 em 22-23 de julho de 2026. Inclui build seguro com Ninja jobs=2, testes nativos do OxtaMem, MQAR com tres sementes de treino nao nulas, bAbI QA1 publico, NSOS+OxtaMem em texto estruturado e extrapolacao MQAR ate 64 pares.

**Atencao:** seed=0 e sentinela nao deterministica no NSOS e foi excluida das campanhas estatisticas. Os outputs preservados sao evidencia da execucao; celulas alteradas para reprodutibilidade tem output limpo.

[Relatorio completo](../docs/benchmarks/jamba_vs_nsos_oxtamem_t4_2026-07-22.md) - [Resultados brutos](../docs/benchmarks/results/expanded_validation_t4_2026-07-22.json)


# OxtaMem â€” prova de qualidade end-to-end (T4)

**O modelo responde melhor quando o OxtaMem lhe entrega o fato certo**, numa tarefa
onde os fatos excedem a janela de contexto? Mesmo modelo (treinado em recall
associativo MQAR). No teste: **modelo-so** (janela recente de C fatos) vs
**modelo+OxtaMem** (recupera o fato via `forward_trunk` e injeta).

Robusto: erros de build aparecem inteiros. A validacao oficial exige OxtaMem
nativo; o fallback numpy fica disponivel apenas para diagnostico. Runtime: **T4**. Rode em ordem.

## 1 â€” Helpers + GPU + clone (branch `nsos-gpu-phases12`)

In [1]:
import os, sys, subprocess, pathlib, glob, shutil, base64
def sh(cmd, cwd=None, extra_env=None, quiet=False):
    """Roda um comando mostrando o ERRO REAL (stdout+stderr) se falhar."""
    env = dict(os.environ);
    if extra_env: env.update(extra_env)
    # string -> roda no shell (necessario p/ pipes como 'curl | sh'); lista -> direto
    r = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True, shell=isinstance(cmd, str))
    if r.returncode != 0:
        shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
        print("COMANDO FALHOU:", shown)
        if r.stdout.strip(): print("--- stdout (fim) ---\n" + r.stdout[-3500:])
        if r.stderr.strip(): print("--- stderr (fim) ---\n" + r.stderr[-3500:])
        raise RuntimeError(f"exit {r.returncode}: {shown}")
    elif not quiet and r.stdout.strip():
        print(r.stdout[-800:])
    return r

print(sh(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], quiet=True).stdout.strip())
REPO = '/content/reimagined'; BRANCH = 'nsos-gpu-phases12'
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/vitorGgC569/reimagined.git')
TOKEN = None
try:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'GH_TOKEN'):
        try:
            TOKEN = userdata.get(key)
            if TOKEN: print('token', key, 'ok'); break
        except Exception: pass
except Exception as e:
    print('userdata indisponivel:', e)
if not TOKEN and 'github.com' in REPO_URL:
    import getpass
    TOKEN = getpass.getpass('GitHub token (entrada oculta): ')
assert TOKEN or 'github.com' not in REPO_URL, 'repo privado: configure o Colab Secret GITHUB_TOKEN (icone chave)'
git_env = {}
if TOKEN:
    auth = base64.b64encode(f'oauth2:{TOKEN}'.encode()).decode()
    git_env = {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader',
               'GIT_CONFIG_VALUE_0': f'Authorization: Basic {auth}'}
if not pathlib.Path(REPO, '.git').exists():
    if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
    sh(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO], extra_env=git_env)
else:
    sh(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', BRANCH], extra_env=git_env)
    sh(['git', '-C', REPO, 'reset', '--hard', 'FETCH_HEAD'])
commit = sh(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], quiet=True).stdout.strip()
os.environ['REPO'] = REPO
print('repo pronto:', REPO, '| branch', BRANCH, '| commit', commit)

Tesla T4, 15360 MiB
GitHub token (entrada oculta): Â·Â·Â·Â·Â·Â·Â·Â·Â·Â·
repo pronto: /content/reimagined | branch nsos-gpu-phases12 | commit 4462b62


## 2 â€” Build do `nsos_ext` (CUDA â€” receita do bootstrap oficial)

In [3]:
import os, sys, subprocess, pathlib, glob
REPO = '/content/reimagined'
BUILD = f'{REPO}/OXN/nsos/build-t4'
sh(['apt-get', 'update', '-qq'])
sh(['apt-get', 'install', '-y', '-qq', 'cmake', 'ninja-build', 'build-essential'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'pybind11>=2.10', 'numpy'])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], capture_output=True, text=True).stdout
arch = next((v for k, v in {'T4':'75','V100':'70','L4':'89','A100':'80','H100':'90'}.items() if k in gpu), '75')
print('GPU', gpu.strip(), '-> sm_' + arch)
sh(['cmake', '-G', 'Ninja', '-S', f'{REPO}/OXN/nsos', '-B', BUILD,
    '-DCMAKE_BUILD_TYPE=Release', '-DNSOS_ENABLE_CUDA=ON', '-DNSOS_BUILD_PYTHON=ON',
    '-DNSOS_BUILD_TESTS=OFF', '-DNSOS_BUILD_CLI=OFF', '-DNSOS_BUILD_API=OFF',
    '-DNSOS_BUILD_OXTAMEM=OFF', f'-DNSOS_CUDA_ARCHITECTURES={arch}'])
cmd = ['cmake', '--build', BUILD, '--parallel', '2', '--target', 'nsos_ext']
print('build seguro:', ' '.join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print((r.stdout + '\n' + r.stderr)[-5000:])
assert r.returncode == 0, f'build seguro falhou: {r.returncode}'
so = glob.glob(f'{BUILD}/**/nsos_ext*.so', recursive=True)
assert so, 'nsos_ext nao compilou'
sys.path.insert(0, str(pathlib.Path(so[0]).parent))
import nsos_ext as nsos
print('nsos_ext OK:', so[0], '| jobs=2')


build seguro: cmake --build /content/reimagined/OXN/nsos/build-t4 --parallel 2 --target nsos_ext
[1/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/mamba_kernels.cu.o
[2/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/kernels.cu.o
/content/reimagined/OXN/nsos/include/cuda/bitnet_math.cuh(142): warning #177-D: variable "acc" was declared but never referenced
      int acc = 0;
          ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

[3/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/bitnet_kernels.cu.o
[4/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/moe_kernels.cu.o
[5/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/kan_kernels.cu.o
[6/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/attention_train_kernels.cu.o
[7/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/fused_optimizer_kernels.cu.o
[8/11] Building CUDA object CMakeFiles/nsos_core.dir/src/cuda/sparse_attention_kernel

## 3 â€” Memoria vetorial: OxtaMem nativo (rust+maturin)

A validacao oficial exige o OxtaMem nativo. O store numpy com retrieval cosine
identico permanece apenas como ferramenta diagnostica (`REQUIRE_NATIVE = False`).

In [4]:
import numpy as np
class NumpyMemStore:
    """Fallback de memoria vetorial: cosine KNN exato. Mesma interface do
    PyGeodesicEngine usada aqui (write_with_vector / search_similar / flush)."""
    def __init__(self): self.vecs=[]; self.values=[]
    def set_sync_on_write(self, b): pass
    def flush(self): pass
    def write_with_vector(self, token_id, value, vector):
        v = np.asarray(vector, dtype=np.float32); v = v/(np.linalg.norm(v)+1e-8)
        self.vecs.append(v); self.values.append(bytes(value)); return len(self.values)-1
    def search_similar(self, vector, k):
        if not self.vecs: return []
        q = np.asarray(vector, dtype=np.float32); q = q/(np.linalg.norm(q)+1e-8)
        sims = np.stack(self.vecs) @ q
        idx = np.argsort(-sims)[:k]
        return [self.values[i] for i in idx]

REQUIRE_NATIVE = True
USE_OXTA = False; PyGeodesicEngine = None
try:
    sh(['apt-get', 'install', '-y', '-qq', 'patchelf'])   # maturin repair dep
    if shutil.which('cargo') is None:
        print('instalando rust toolchain...')
        sh('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal')
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'maturin'])
    # --compatibility linux evita o repair manylinux (que exige patchelf/docker e quebra no Colab)
    sh(['maturin', 'build', '--release', '--compatibility', 'linux', '--interpreter', sys.executable],
       cwd=f'{REPO}/modules/oxtamem')
    whl = glob.glob(f'{REPO}/modules/oxtamem/oxta_engine/target/wheels/oxta_mem-*.whl')
    assert whl, 'wheel nao gerado'
    sh([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', whl[0]])
    from oxta_mem.native import PyGeodesicEngine
    USE_OXTA = True
    print('>>> OxtaMem NATIVO OK:', whl[0])
except Exception as e:
    print('>>> OxtaMem nativo indisponivel. Motivo:', repr(e)[:400])
    if REQUIRE_NATIVE:
        raise RuntimeError('Validacao oficial exige OxtaMem nativo; fallback numpy nao e prova do OxtaMem') from e
print('memoria em uso:', 'OxtaMem nativo' if USE_OXTA else 'NumpyMemStore (cosine KNN)')

ge patchelf.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(Reading database ... 40%
(Reading database ... 45%
(Reading database ... 50%
(Reading database ... 55%
(Reading database ... 60%
(Reading database ... 65%
(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122415 files and directories currently installed.)
Preparing to unpack .../patchelf_0.14.3-1_amd64.deb ...
Unpacking patchelf (0.14.3-1) ...
Setting up patchelf (0.14.3-1) ...
Processing triggers for man-db (2.10.2-1) ...

instalando rust toolchain...

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current sh

In [4]:
sh(['cargo', 'test', '--release', '--locked'], cwd=f'{REPO}/modules/oxtamem/oxta_engine')

import gc, tempfile
root_gate = tempfile.mkdtemp(prefix='oxta_gate_', dir='/content')
db_gate = os.path.join(root_gate, 'store.db')
gate = PyGeodesicEngine(db_gate, 8)
gate.set_sync_on_write(False)
vectors_gate = []
for i in range(10):
    v = np.zeros(128, dtype=np.float32); v[i] = 1.0
    vectors_gate.append(v.tolist())
    gate.write_with_vector(str(i), bytes([i]), v.tolist())
gate.flush()
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect()

gate = PyGeodesicEngine(db_gate, 8)
assert all(gate.read_latest(str(i)) == bytes([i]) for i in range(10))
assert all(gate.search_similar(vectors_gate[i], 1)[0] == bytes([i]) for i in range(10))
del gate; gc.collect(); shutil.rmtree(root_gate, ignore_errors=True)
print('>>> OxtaMem alignment/persistence gate: 10/10 OK')

open ... ok
test engine::tests::write_throughput_sync_vs_deferred ... ignored, write-throughput benchmark; run explicitly with --ignored --nocapture
test server::tests::parse_resp_never_panics_on_arbitrary_input ... ok
test server::tests::response_builders_enforce_total_limit ... ok
test sharding::tests::empty_cluster_falls_back_to_local ... ok
test sharding::tests::rendezvous_hashing_is_stable_for_same_key ... ok
test engine::tests::arena_grows_beyond_initial_size ... ok

test result: ok. 19 passed; 0 failed; 2 ignored; 0 measured; 0 filtered out; finished in 1.65s


running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s


>>> OxtaMem alignment/persistence gate: 10/10 OK


## 4 â€” Tarefa de memoria (MQAR) + treino

Modelo aprende lookup associativo em contexto (`k1 v1 ... [Q] kq` -> `vq`), com
comprimento variavel 1..8 -> robusto tanto ao 1 fato injetado quanto a janela cheia.

In [ ]:
import numpy as np, random, math, time
GPU = nsos.Device.GPU
os.environ['NSOS_ATTN_LAST_GUARD'] = '0'   # hibrido [M,A,M,A] (atencao ajuda recall)
QUERY = 1; V0, V1 = 2, 66; K0, K1 = 66, 1266; VOCAB = 1266   # values byte-safe (<256); muitas keys
D_MODEL, D_STATE = 128, 64; N_PAIRS = 8; BATCH = 32; LR = 2e-3; STEPS = 2500; SEED = 2

def _set(c, k, v):
    try: setattr(c, k, v)
    except Exception: pass
def build():
    c = nsos.ModelConfig(); c.num_layers=4; c.d_model=D_MODEL; c.vocab_size=VOCAB
    c.n_heads=4; c.n_kv_heads=2; c.max_context_tokens=1024
    c.use_moe=False; c.use_kan=False; c.use_ttt=False; c.dropout=0.0
    _set(c,'mamba2_faithful',True); _set(c,'mamba_expand',2); _set(c,'mamba_head_dim',64)
    _set(c,'mamba_state_expansion',D_STATE); _set(c,'mamba_n_groups',1)
    c.attention_period=2; c.attention_slot=1
    m = nsos.JambaModel(c, GPU); m.to(GPU); return m

def mqar_seq(rng, n_pairs):
    keys = rng.sample(range(K0, K1), n_pairs)
    vals = [rng.randrange(V0, V1) for _ in range(n_pairs)]
    seq = []
    for k, v in zip(keys, vals): seq += [k, v]
    qi = rng.randrange(n_pairs); seq += [QUERY, keys[qi]]
    return seq, vals[qi]

def batch(rng, n_pairs):
    ps, an = [], []
    for _ in range(BATCH):
        s, v = mqar_seq(rng, n_pairs); ps.append(s); an.append([v])
    return ps, an

nsos.set_seed(SEED)
if hasattr(nsos, 'set_deterministic_reductions'):
    nsos.set_deterministic_reductions(False)
model = build()
tr = nsos.Trainer(model, LR); tr.warmup_steps=100; tr.total_training_steps=STEPS
tr.phase_scheduler.progressive_qat_enabled=False
rng = random.Random(7); model.set_training_mode(True); t0=time.time(); run=0.0
for s in range(STEPS):
    npair = rng.randint(1, N_PAIRS)
    ps, an = batch(rng, npair); run += tr.train_supervised_batch(ps, an)
    if (s+1) % 500 == 0: print(f'  step {s+1}/{STEPS} loss~{run/500:.3f}'); run=0.0
model.set_training_mode(False)
print(f'treino ok em {time.time()-t0:.0f}s')

## 5 â€” Sanity: o modelo aprendeu o lookup em contexto?

In [9]:
def _argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())
vr = random.Random(999); ok = 0; T = 300
for _ in range(T):
    s, v = mqar_seq(vr, N_PAIRS); ok += (_argmax_last(s) == v)
acc = ok / T
print(f'MQAR in-context acc (n_pairs={N_PAIRS}) = {acc:.3f}')
assert acc > 0.6, 'modelo nao aprendeu o lookup â€” aumente STEPS ou cheque o build'

MQAR in-context acc (n_pairs=8) = 0.920


## 6 â€” Prova end-to-end: modelo-so vs modelo+OxtaMem (fatos >> contexto)

In [6]:
def argmax_last(ids):
    lg = np.asarray(model.forward_ids(ids).cpu().numpy()).reshape(-1, VOCAB)
    return int(lg[-1].argmax())

def embed_key(k):
    tr_ = np.asarray(model.forward_trunk([QUERY, k]).cpu().numpy()).reshape(-1, D_MODEL)
    return tr_[-1].astype(np.float32).tolist()

def make_store():
    if USE_OXTA:
        import tempfile
        root = tempfile.mkdtemp(prefix='oxta_qa_', dir='/content')
        db = os.path.join(root, 'qa_store.db')
        eng = PyGeodesicEngine(db, 8); eng.set_sync_on_write(False); return eng, root
    return NumpyMemStore(), None

def run_eval(N, C=8, n_query=200, seed=0):
    rng = random.Random(seed)
    keys = rng.sample(range(K0, K1), min(N, K1 - K0)); N = len(keys)
    vals = {k: rng.randrange(V0, V1) for k in keys}
    store, store_root = make_store()
    for k in keys:
        store.write_with_vector(f'k{k}', bytes([vals[k]]), embed_key(k))
    store.flush()
    order = keys[:]
    qrng = random.Random(seed * 7 + 1)
    qs = [qrng.randrange(N) for _ in range(n_query)]
    alone = oxta = retr = 0
    for qi in qs:
        qk = order[qi]; true_v = vals[qk]
        window = order[max(0, N - C):]
        seq = []
        for k in window: seq += [k, vals[k]]
        seq += [QUERY, qk]
        alone += (argmax_last(seq) == true_v)
        got = store.search_similar(embed_key(qk), 1)
        rv = got[0][0] if got else -1
        retr += (rv == true_v)
        oxta += (argmax_last([qk, rv, QUERY, qk]) == true_v)
    result = dict(N=N, C=C, alone=alone/len(qs), oxta=oxta/len(qs), retr=retr/len(qs))
    if store_root:
        import gc
        del store; gc.collect(); shutil.rmtree(store_root, ignore_errors=True)
    return result

print(f'store: {"OxtaMem nativo" if USE_OXTA else "NumpyMemStore"}')
print(f'{"N":>6} {"C(janela)":>10} {"modelo-so":>11} {"modelo+OxtaMem":>15} {"(retr@1)":>10}')
results = []
for N in (50, 200, 1000):
    r = run_eval(N, C=8)
    results.append(r)
    print(f'{r["N"]:>6} {r["C"]:>10} {r["alone"]:>11.3f} {r["oxta"]:>15.3f} {r["retr"]:>10.3f}')
assert USE_OXTA, 'validacao oficial exige OxtaMem nativo'
assert min(r['retr'] for r in results) >= 0.99, 'recall exato do OxtaMem abaixo de 99%'
assert results[-1]['oxta'] > results[-1]['alone'] + 0.30, 'OxtaMem nao melhorou o caso N=1000'

store: OxtaMem nativo
     N  C(janela)   modelo-so  modelo+OxtaMem   (retr@1)
    50          8       0.150           1.000      1.000
   200          8       0.060           1.000      1.000
  1000          8       0.025           1.000      1.000


In [10]:
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
from transformers import JambaConfig, JambaForCausalLM
print('Jamba HF import OK')

torch 2.11.0+cu128 | transformers 5.13.1
Jamba HF import OK


In [11]:
hf_cfg = JambaConfig(
    vocab_size=VOCAB, hidden_size=D_MODEL, intermediate_size=4*D_MODEL,
    num_hidden_layers=4, num_attention_heads=4, num_key_value_heads=2,
    attn_layer_period=2, attn_layer_offset=1,
    mamba_d_state=D_STATE, mamba_d_conv=4, mamba_expand=2,
    num_experts=1, num_experts_per_tok=1,
    hidden_act='silu', attention_dropout=0.0,
    rms_norm_eps=1e-5, use_cache=False, use_mamba_kernels=False,
    tie_word_embeddings=True, max_position_embeddings=1024
)
torch.manual_seed(SEED)
hf_probe = JambaForCausalLM(hf_cfg).cuda().float()
hf_probe.eval()
with torch.no_grad():
    z = hf_probe(torch.tensor([[66,2,QUERY,66]], device='cuda'), use_cache=False).logits
print('HF Jamba probe', tuple(z.shape), '| params', f'{sum(p.numel() for p in hf_probe.parameters()):,}')
print('camadas:', [type(x).__name__ for x in hf_probe.model.layers])
del z, hf_probe; torch.cuda.empty_cache()

[transformers] The fast path is not available because on of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d.


HF Jamba probe (1, 4, 1266) | params 1,354,896
camadas: ['JambaMambaDecoderLayer', 'JambaAttentionDecoderLayer', 'JambaMambaDecoderLayer', 'JambaAttentionDecoderLayer']


In [12]:
import gc, time, math, random, json
import numpy as np
import torch.nn.functional as F

# seed=0 e sentinela nao deterministica no NSOS (tensor.cpp usa random_device).
# A campanha oficial usa apenas sementes nao nulas.
TRAIN_SEEDS = [1, 2, 3]
MS_EVAL_TOTAL = 600
MS_EVAL_PAIRS = (1, 4, 8)

# Mesmo conjunto de teste, nunca visto no treino, para todos os modelos/sementes.
ms_eval_sets = {}
for n_pair in MS_EVAL_PAIRS:
    rr = random.Random(70000 + n_pair)
    pairs = [mqar_seq(rr, n_pair) for _ in range(MS_EVAL_TOTAL)]
    ms_eval_sets[n_pair] = ([x[0] for x in pairs], [x[1] for x in pairs])

def ms_nsos_acc(m, n_pair):
    seqs, ys = ms_eval_sets[n_pair]
    correct = 0
    for seq, y in zip(seqs, ys):
        lg = np.asarray(m.forward_ids(seq).cpu().numpy()).reshape(-1, VOCAB)
        correct += int(int(lg[-1].argmax()) == y)
    return correct / len(ys)

@torch.no_grad()
def ms_hf_acc(m, n_pair, chunk=128):
    seqs, ys = ms_eval_sets[n_pair]
    pred = []
    for start in range(0, len(seqs), chunk):
        ids = torch.tensor(seqs[start:start+chunk], device='cuda', dtype=torch.long)
        pred.extend(m(ids, use_cache=False).logits[:, -1, :].argmax(-1).cpu().tolist())
    return sum(a == b for a, b in zip(pred, ys)) / len(ys)

def ms_lr_at(step):
    if step <= 100:
        return LR * step / 100.0
    progress = min(max((step - 100) / (STEPS - 100), 0.0), 1.0)
    return LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

hf_moe_cfg_ms = JambaConfig(
    vocab_size=VOCAB, hidden_size=D_MODEL, intermediate_size=4*D_MODEL,
    num_hidden_layers=4, num_attention_heads=4, num_key_value_heads=2,
    attn_layer_period=2, attn_layer_offset=1,
    mamba_d_state=D_STATE, mamba_d_conv=4, mamba_expand=2,
    num_experts=16, num_experts_per_tok=2,
    expert_layer_period=2, expert_layer_offset=1,
    router_aux_loss_coef=0.001, output_router_logits=True,
    hidden_act='silu', attention_dropout=0.0,
    rms_norm_eps=1e-5, use_cache=False, use_mamba_kernels=False,
    tie_word_embeddings=True, max_position_embeddings=1024
)

def ms_train_nsos(seed):
    nsos.set_seed(seed)
    if hasattr(nsos, 'set_deterministic_reductions'):
        nsos.set_deterministic_reductions(True)
    m = build()
    trainer = nsos.Trainer(m, LR)
    trainer.warmup_steps = 100
    trainer.total_training_steps = STEPS
    trainer.phase_scheduler.progressive_qat_enabled = False
    rr = random.Random(100000 + seed)
    m.set_training_mode(True)
    t0s = time.time()
    run = 0.0
    for step in range(STEPS):
        n_pair = rr.randint(1, N_PAIRS)
        ps, an = batch(rr, n_pair)
        run += trainer.train_supervised_batch(ps, an)
        if (step + 1) % 500 == 0:
            print(f' [NSOS seed={seed}] {step+1}/{STEPS} loss~{run/500:.3f}')
            run = 0.0
    m.set_training_mode(False)
    elapsed = time.time() - t0s
    scores = {n: ms_nsos_acc(m, n) for n in MS_EVAL_PAIRS}
    params = sum(int(p.data.size) for p in m.parameters())
    return m, dict(seed=seed, data_seed=100000+seed, seconds=elapsed,
                   params=params, scores=scores)

def ms_train_hf(seed, cfg, label, use_moe=False):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    m = JambaForCausalLM(cfg).cuda().float()
    opt = torch.optim.AdamW(m.parameters(), lr=LR, betas=(0.9, 0.999),
                            eps=1e-8, weight_decay=0.01)
    rr = random.Random(100000 + seed)
    m.train()
    t0s = time.time()
    run_ce = 0.0
    for step in range(STEPS):
        n_pair = rr.randint(1, N_PAIRS)
        seqs, ys = [], []
        for _ in range(BATCH):
            seq, value = mqar_seq(rr, n_pair)
            seqs.append(seq)
            ys.append(value)
        ids = torch.tensor(seqs, device='cuda', dtype=torch.long)
        target = torch.tensor(ys, device='cuda', dtype=torch.long)
        for group in opt.param_groups:
            group['lr'] = ms_lr_at(step)
        out = m(ids, use_cache=False, output_router_logits=use_moe)
        ce = F.cross_entropy(out.logits[:, -1, :], target)
        loss = ce
        if use_moe:
            loss = loss + cfg.router_aux_loss_coef * out.aux_loss
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        opt.step()
        run_ce += float(ce.detach())
        if (step + 1) % 500 == 0:
            print(f' [{label} seed={seed}] {step+1}/{STEPS} ce~{run_ce/500:.3f}')
            run_ce = 0.0
    elapsed = time.time() - t0s
    m.eval()
    scores = {n: ms_hf_acc(m, n) for n in MS_EVAL_PAIRS}
    params = sum(p.numel() for p in m.parameters())
    return m, dict(seed=seed, data_seed=100000+seed, seconds=elapsed,
                   params=params, scores=scores)

# Libera o treino seed=0 usado somente na auditoria de nao determinismo.
if 'model' in globals():
    del model
if 'tr' in globals():
    del tr
gc.collect()
if hasattr(nsos, 'release_cached_memory'):
    nsos.release_cached_memory()
torch.cuda.empty_cache()

multitrain_results = {'nsos': [], 'hf_dense': [], 'hf_moe': []}
for seed in TRAIN_SEEDS:
    print()
    print('=== TRAIN SEED', seed, '===')
    ns_m, rec = ms_train_nsos(seed)
    multitrain_results['nsos'].append(rec)
    print(' NSOS RESULT', rec)
    del ns_m
    gc.collect()
    if hasattr(nsos, 'release_cached_memory'):
        nsos.release_cached_memory()

    dense_m, rec = ms_train_hf(seed, hf_cfg, 'HF-dense', use_moe=False)
    multitrain_results['hf_dense'].append(rec)
    print(' HF DENSE RESULT', rec)
    del dense_m
    gc.collect()
    torch.cuda.empty_cache()

    moe_m, rec = ms_train_hf(seed, hf_moe_cfg_ms, 'HF-MoE', use_moe=True)
    multitrain_results['hf_moe'].append(rec)
    print(' HF MOE RESULT', rec)
    del moe_m
    gc.collect()
    torch.cuda.empty_cache()

print()
print('MULTIPLAS SEMENTES DE TREINO: media +- DP amostral; IC95% t(df=2)')
print(f'{"pares":>7} {"HF denso":>26} {"HF MoE":>26} {"NSOS":>26}')
multitrain_summary = {}
tcrit = 4.3026527299
for arm in ('hf_dense', 'hf_moe', 'nsos'):
    multitrain_summary[arm] = {}
    for n in MS_EVAL_PAIRS:
        vals = [x['scores'][n] for x in multitrain_results[arm]]
        mean = float(np.mean(vals))
        std = float(np.std(vals, ddof=1))
        half = float(tcrit * std / math.sqrt(len(vals)))
        multitrain_summary[arm][n] = dict(mean=mean, std=std,
                                          ci95=[max(0.0, mean-half), min(1.0, mean+half)],
                                          runs=vals)
for n in MS_EVAL_PAIRS:
    d = multitrain_summary['hf_dense'][n]
    e = multitrain_summary['hf_moe'][n]
    s = multitrain_summary['nsos'][n]
    print(f'{n:>7} {d["mean"]:>7.3f} +- {d["std"]:<7.3f} {e["mean"]:>7.3f} +- {e["std"]:<7.3f} {s["mean"]:>7.3f} +- {s["std"]:<7.3f}')

multitrain_times = {
    arm: dict(mean=float(np.mean([x['seconds'] for x in rows])),
              std=float(np.std([x['seconds'] for x in rows], ddof=1)),
              runs=[x['seconds'] for x in rows])
    for arm, rows in multitrain_results.items()
}
multitrain_artifact = dict(
    protocol=dict(train_seeds=TRAIN_SEEDS, eval_examples_per_difficulty=MS_EVAL_TOTAL,
                  eval_pairs=list(MS_EVAL_PAIRS), steps=STEPS, batch=BATCH, lr=LR,
                  note_seed_zero='NSOS seed=0 is nondeterministic sentinel; excluded',
                  nsos_deterministic_reductions=True),
    results=multitrain_results, summary=multitrain_summary, times=multitrain_times)
print('TEMPOS (informativos)', multitrain_times)
print('RESULT_JSON_BEGIN')
print(json.dumps(multitrain_artifact, sort_keys=True))
print('RESULT_JSON_END')
assert all(len(multitrain_results[a]) == 3 for a in multitrain_results)
assert all(np.isfinite(multitrain_summary[a][8]['mean']) for a in multitrain_summary)
print('MULTITRAIN_NONZERO_3_SEEDS: APROVADO')


=== TRAIN SEED 1 ===
 [NSOS seed=1] 500/2500 loss~8.733
 [NSOS seed=1] 1000/2500 loss~5.726
 [NSOS seed=1] 1500/2500 loss~5.080
 [NSOS seed=1] 2000/2500 loss~3.869
 [NSOS seed=1] 2500/2500 loss~3.224
 NSOS RESULT {'seed': 1, 'data_seed': 100001, 'seconds': 275.83758759498596, 'params': 768152, 'scores': {1: 1.0, 4: 0.5816666666666667, 8: 0.28833333333333333}}
 [HF-dense seed=1] 500/2500 ce~3.614
 [HF-dense seed=1] 1000/2500 ce~2.648
 [HF-dense seed=1] 1500/2500 ce~2.393
 [HF-dense seed=1] 2000/2500 ce~2.152
 [HF-dense seed=1] 2500/2500 ce~2.079
 HF DENSE RESULT {'seed': 1, 'data_seed': 100001, 'seconds': 110.47886419296265, 'params': 1354896, 'scores': {1: 0.9983333333333333, 4: 0.25166666666666665, 8: 0.12833333333333333}}
 [HF-MoE seed=1] 500/2500 ce~3.627
 [HF-MoE seed=1] 1000/2500 ce~2.600
 [HF-MoE seed=1] 1500/2500 ce~2.339
 [HF-MoE seed=1] 2000/2500 ce~2.127
 [HF-MoE seed=1] 2500/2500 ce~2.059
 HF MOE RESULT {'seed': 1, 'data_seed': 100001, 'seconds': 145.5970618724823, 'params'

## 9 â€” MÃºltiplas sementes de treino (protocolo corrigido)

**Achado de reprodutibilidade:** no NSOS, seed=0 nÃ£o Ã© uma semente vÃ¡lida: Ã© o sentinela de modo nÃ£o determinÃ­stico. Em tensor.cpp, esse caminho inicializa o gerador com std::random_device. Por isso, as rodadas antigas nominalmente chamadas de seed 0 nÃ£o devem ser tratadas como repetÃ­veis.

**Protocolo oficial desta rodada:** sementes de treino 1, 2 e 3; 2.500 passos; batch 32; LR 0,002; mesmo fluxo de dados por semente entre os braÃ§os; teste fixo separado com 600 exemplos por dificuldade; NSOS com reduÃ§Ãµes determinÃ­sticas. MÃ©dia Â± desvio-padrÃ£o amostral entre trÃªs treinos; IC95% por t de Student com 2 graus de liberdade.

| Pares | Jamba oficial denso | Jamba oficial MoE | NSOS |
|---:|---:|---:|---:|
| 1 | 0,999 Â± 0,001 | 0,999 Â± 0,001 | 1,000 Â± 0,000 |
| 4 | 0,301 Â± 0,076 | 0,361 Â± 0,137 | 0,846 Â± 0,229 |
| 8 | 0,146 Â± 0,028 | 0,157 Â± 0,054 | 0,694 Â± 0,354 |

**ExecuÃ§Ãµes individuais do NSOS:** em 8 pares, 0,288 / 0,937 / 0,858. A mÃ©dia supera claramente os dois rivais neste orÃ§amento, mas a variÃ¢ncia Ã© alta e o IC95% Ã© largo. A leitura honesta Ã©: **vantagem forte e promissora em MQAR sintÃ©tico, ainda com instabilidade de otimizaÃ§Ã£o que precisa ser investigada**.

Os tempos desta rodada sÃ£o apenas informativos: o NSOS usou o caminho de reduÃ§Ãµes determinÃ­sticas, muito mais lento que o caminho GPU rÃ¡pido. NÃ£o usar esses tempos para alegar velocidade relativa.


In [16]:
import pandas as pd, hashlib, urllib.request, pathlib
BABI_REV = '1d86ad39d1c3ea2ff4b77eeb85f7c6ebd622a95f'
BABI_BASE = f'https://huggingface.co/datasets/facebook/babi_qa/resolve/{BABI_REV}/en-10k-qa1'
babi_paths = {}
for split in ('train', 'test'):
    url = f'{BABI_BASE}/{split}/0000.parquet?download=true'
    path = pathlib.Path(f'/content/babi_qa1_{split}.parquet')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        blob = resp.read()
    path.write_bytes(blob)
    babi_paths[split] = path
    print(split, 'bytes', len(blob), 'sha256', hashlib.sha256(blob).hexdigest())
babi_train_df = pd.read_parquet(babi_paths['train'])
babi_test_df = pd.read_parquet(babi_paths['test'])
print('rows', len(babi_train_df), len(babi_test_df))
print('columns', babi_train_df.columns.tolist())
print('sample', babi_train_df.iloc[0].to_dict())

train bytes 51147 sha256 f1d67aa230d7aba0ed310df0d696a3ba9a07270e1670fe64c6901c24e5016f3f
test bytes 9410 sha256 9875dfad271cbfa5c49adb5809dd67be3826394a5d4d66dc74cab0c81483e2f8
rows 2000 200
columns ['story']
sample {'story': {'id': array(['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', '15'], dtype=object), 'type': array([0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1]), 'text': array(['Mary moved to the bathroom.', 'John went to the hallway.',
       'Where is Mary?', 'Daniel went back to the hallway.',
       'Sandra moved to the garden.', 'Where is Daniel?',
       'John moved to the office.', 'Sandra journeyed to the bathroom.',
       'Where is Daniel?', 'Mary moved to the hallway.',
       'Daniel travelled to the office.', 'Where is Daniel?',
       'John went back to the garden.', 'John moved to the bedroom.',
       'Where is Sandra?'], dtype=object), 'supporting_ids': array([array([], dtype=object), array([], dtype=object),
       array(['1'],

In [17]:
import re, collections, numpy as np

def unpack_babi(df):
    out = []
    for cell in df['story']:
        context = []
        for typ, text, answer, supporting in zip(cell['type'], cell['text'], cell['answer'], cell['supporting_ids']):
            if int(typ) == 0:
                context.append(str(text))
            else:
                out.append(dict(context=list(context), question=str(text), answer=str(answer),
                                supporting=[str(x) for x in supporting]))
    return out

babi_train_ex = unpack_babi(babi_train_df)
babi_test_ex = unpack_babi(babi_test_df)
TOKEN_RE = re.compile(r"[a-z]+|[?.]", re.I)
def words(s):
    return TOKEN_RE.findall(s.lower())

special = ['<pad>', '<unk>', '<sep>', '<answer>']
train_tokens = set()
for ex in babi_train_ex:
    for sent in ex['context']:
        train_tokens.update(words(sent))
    train_tokens.update(words(ex['question']))
    train_tokens.add(ex['answer'].lower())
babi_vocab_list = special + sorted(train_tokens)
babi_stoi = {w:i for i,w in enumerate(babi_vocab_list)}

def encode_babi(ex):
    ids = []
    for sent in ex['context']:
        ids += [babi_stoi.get(w, 1) for w in words(sent)] + [babi_stoi['<sep>']]
    ids += [babi_stoi.get(w, 1) for w in words(ex['question'])] + [babi_stoi['<answer>']]
    return ids, babi_stoi.get(ex['answer'].lower(), 1)

babi_train_encoded = [encode_babi(x) for x in babi_train_ex]
babi_test_encoded = [encode_babi(x) for x in babi_test_ex]
lengths_train = [len(x[0]) for x in babi_train_encoded]
lengths_test = [len(x[0]) for x in babi_test_encoded]
BABI_MAXLEN = max(max(lengths_train), max(lengths_test))
BABI_VOCAB = len(babi_vocab_list)
BABI_PAD = babi_stoi['<pad>']

def left_pad(ids):
    return [BABI_PAD] * (BABI_MAXLEN - len(ids)) + ids

babi_train_padded = [(left_pad(x), y) for x,y in babi_train_encoded]
babi_test_padded = [(left_pad(x), y) for x,y in babi_test_encoded]
counts = collections.Counter(y for _,y in babi_test_padded)
majority = max(counts.values()) / len(babi_test_padded)
unk_test = sum(tok == babi_stoi['<unk>'] for seq,_ in babi_test_padded for tok in seq)
print('examples train/test', len(babi_train_padded), len(babi_test_padded))
print('vocab', BABI_VOCAB, 'maxlen', BABI_MAXLEN,
      'train len p50/p95/max', np.percentile(lengths_train,[50,95,100]).tolist())
print('answer labels', {babi_vocab_list[k]:v for k,v in sorted(counts.items())})
print('majority baseline', majority, 'UNK tokens test', unk_test)
print('encoded sample tail', [babi_vocab_list[t] for t in babi_train_padded[0][0] if t != BABI_PAD],
      '=>', babi_vocab_list[babi_train_padded[0][1]])

examples train/test 10000 1000
vocab 25 maxlen 83 train len p50/p95/max [48.0, 78.0, 83.0]
answer labels {'bathroom': 149, 'bedroom': 171, 'garden': 187, 'hallway': 154, 'kitchen': 157, 'office': 182}
majority baseline 0.187 UNK tokens test 0
encoded sample tail ['mary', 'moved', 'to', 'the', 'bathroom', '.', '<sep>', 'john', 'went', 'to', 'the', 'hallway', '.', '<sep>', 'where', 'is', 'mary', '?', '<answer>'] => bathroom


In [18]:
import gc, time, math, random, numpy as np
import torch
import torch.nn.functional as F

BABI_SEED = 11
BABI_STEPS = 1500
BABI_BATCH = 32
BABI_LR = 2e-3
BABI_DMODEL = 128
BABI_DSTATE = 64

def babi_build_nsos(seed):
    nsos.set_seed(seed)
    if hasattr(nsos, 'set_deterministic_reductions'):
        nsos.set_deterministic_reductions(False)
    c = nsos.ModelConfig()
    c.num_layers=4; c.d_model=BABI_DMODEL; c.vocab_size=BABI_VOCAB
    c.n_heads=4; c.n_kv_heads=2; c.max_context_tokens=128
    c.use_moe=False; c.use_kan=False; c.use_ttt=False; c.dropout=0.0
    _set(c,'mamba2_faithful',True); _set(c,'mamba_expand',2); _set(c,'mamba_head_dim',64)
    _set(c,'mamba_state_expansion',BABI_DSTATE); _set(c,'mamba_n_groups',1)
    c.attention_period=2; c.attention_slot=1
    m = nsos.JambaModel(c, GPU); m.to(GPU)
    return m

def babi_lr(step):
    if step <= 100: return BABI_LR * step / 100.0
    p = min(max((step-100)/(BABI_STEPS-100),0.0),1.0)
    return BABI_LR * (0.1 + 0.9*0.5*(1.0+math.cos(math.pi*p)))

def babi_batch(rr):
    idx = [rr.randrange(len(babi_train_padded)) for _ in range(BABI_BATCH)]
    return [babi_train_padded[i][0] for i in idx], [[babi_train_padded[i][1]] for i in idx]

def babi_test_nsos(m):
    ok = 0
    conf = np.zeros((6,6), dtype=np.int64)
    labels = sorted(set(y for _,y in babi_test_padded))
    li = {v:i for i,v in enumerate(labels)}
    for seq,y in babi_test_padded:
        lg = np.asarray(m.forward_ids(seq).cpu().numpy()).reshape(-1,BABI_VOCAB)
        p = int(lg[-1].argmax())
        ok += int(p==y)
        if p in li: conf[li[y],li[p]] += 1
    return ok/len(babi_test_padded), conf, [babi_vocab_list[x] for x in labels]

# NSOS, uma semente exploratoria publica.
nsos.set_seed(BABI_SEED)
babi_nsos = babi_build_nsos(BABI_SEED)
babi_tr = nsos.Trainer(babi_nsos, BABI_LR)
babi_tr.warmup_steps=100; babi_tr.total_training_steps=BABI_STEPS
babi_tr.phase_scheduler.progressive_qat_enabled=False
rr = random.Random(200000+BABI_SEED)
babi_nsos.set_training_mode(True)
t0=time.time(); run=0.0
for step in range(BABI_STEPS):
    ps, an = babi_batch(rr)
    run += babi_tr.train_supervised_batch(ps, an)
    if (step+1)%300==0:
        print(f'[bAbI NSOS] {step+1}/{BABI_STEPS} loss~{run/300:.4f}')
        run=0.0
babi_nsos.set_training_mode(False)
babi_nsos_s=time.time()-t0
babi_nsos_acc, babi_nsos_conf, babi_label_names = babi_test_nsos(babi_nsos)
print('BABI NSOS RESULT', {'seed':BABI_SEED,'seconds':babi_nsos_s,'accuracy':babi_nsos_acc,
                           'params':sum(int(p.data.size) for p in babi_nsos.parameters())})
print('labels', babi_label_names)
print('confusion rows=true cols=pred')
print(babi_nsos_conf)
print('BABI_QA1_NSOS_EXPLORATORY_DONE')

[bAbI NSOS] 300/1500 loss~3.8108
[bAbI NSOS] 600/1500 loss~2.7407
[bAbI NSOS] 900/1500 loss~1.6962
[bAbI NSOS] 1200/1500 loss~1.5834
[bAbI NSOS] 1500/1500 loss~1.5349
BABI NSOS RESULT {'seed': 11, 'seconds': 93.6502742767334, 'accuracy': 0.767, 'params': 609304}
labels ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office']
confusion rows=true cols=pred
[[119   7  11   5   3   4]
 [  9 132   8   7   8   7]
 [  6  11 149   3   9   9]
 [  5   8  10 116   5  10]
 [  9   5  11   6 118   8]
 [ 14   7   9   9  10 133]]
BABI_QA1_NSOS_EXPLORATORY_DONE


In [20]:
def babi_train_hf_logged(cfg,label,use_moe,seed=BABI_SEED,steps=BABI_STEPS):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    m=JambaForCausalLM(cfg).cuda().float()
    opt=torch.optim.AdamW(m.parameters(),lr=BABI_LR,betas=(0.9,0.999),eps=1e-8,weight_decay=0.01)
    rr=random.Random(200000+seed); m.train(); t0=time.time(); run=0.0
    for step in range(steps):
        idx=[rr.randrange(len(babi_train_padded)) for _ in range(BABI_BATCH)]
        ids=torch.tensor([babi_train_padded[i][0] for i in idx],device='cuda',dtype=torch.long)
        target=torch.tensor([babi_train_padded[i][1] for i in idx],device='cuda',dtype=torch.long)
        for group in opt.param_groups: group['lr']=babi_lr(step)
        out=m(ids,use_cache=False,output_router_logits=use_moe)
        ce=F.cross_entropy(out.logits[:,-1,:],target)
        loss=ce+(cfg.router_aux_loss_coef*out.aux_loss if use_moe else 0.0)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step()
        run+=float(ce.detach())
        if (step+1)%100==0:
            print(f'[bAbI {label}] {step+1}/{steps} ce~{run/100:.4f} elapsed={time.time()-t0:.1f}s')
            torch.save({'step':step+1,'model':m.state_dict(),'optimizer':opt.state_dict(),
                        'seed':seed,'label':label},f'/content/babi_{label}_checkpoint.pt')
            run=0.0
    elapsed=time.time()-t0; m.eval(); acc,conf=babi_test_hf(m)
    rec={'seed':seed,'seconds':elapsed,'accuracy':acc,'params':sum(p.numel() for p in m.parameters()),'steps':steps}
    print('BABI',label,'RESULT',rec); print(conf)
    return m,rec

print('START BABI HF-DENSE CLEAN RUN')
babi_hf_dense,babi_hf_dense_rec=babi_train_hf_logged(babi_hf_dense_cfg,'HF-dense',False)
print('BABI_HF_DENSE_DONE')

START BABI HF-DENSE CLEAN RUN
[bAbI HF-dense] 100/1500 ce~1.9943 elapsed=106.3s
[bAbI HF-dense] 200/1500 ce~1.6169 elapsed=212.7s
[bAbI HF-dense] 300/1500 ce~1.4557 elapsed=318.7s
[bAbI HF-dense] 400/1500 ce~1.3915 elapsed=424.5s
[bAbI HF-dense] 500/1500 ce~1.3598 elapsed=530.5s
[bAbI HF-dense] 600/1500 ce~1.3358 elapsed=636.4s
[bAbI HF-dense] 700/1500 ce~1.2890 elapsed=742.4s
[bAbI HF-dense] 800/1500 ce~1.2823 elapsed=848.3s
[bAbI HF-dense] 900/1500 ce~1.2886 elapsed=954.3s
[bAbI HF-dense] 1000/1500 ce~1.2336 elapsed=1060.2s
[bAbI HF-dense] 1100/1500 ce~1.2072 elapsed=1166.1s
[bAbI HF-dense] 1200/1500 ce~1.1805 elapsed=1272.4s
[bAbI HF-dense] 1300/1500 ce~1.1644 elapsed=1378.5s
[bAbI HF-dense] 1400/1500 ce~1.1345 elapsed=1484.5s
[bAbI HF-dense] 1500/1500 ce~1.1140 elapsed=1590.5s
BABI HF-dense RESULT {'seed': 11, 'seconds': 1590.5745222568512, 'accuracy': 0.493, 'params': 1196048, 'steps': 1500}
[[ 85  16  21   8   8  11]
 [ 22  83  19  17  15  15]
 [ 10  20 106   8  18  25]
 [ 15  15

In [21]:
del babi_hf_dense
gc.collect(); torch.cuda.empty_cache()
print('START BABI HF-MOE CLEAN RUN')
babi_hf_moe,babi_hf_moe_rec=babi_train_hf_logged(babi_hf_moe_cfg,'HF-MoE',True)
print('BABI_HF_MOE_DONE')

START BABI HF-MOE CLEAN RUN
[bAbI HF-MoE] 100/1500 ce~2.0583 elapsed=107.6s
[bAbI HF-MoE] 200/1500 ce~1.8474 elapsed=215.4s
[bAbI HF-MoE] 300/1500 ce~1.8168 elapsed=323.1s
[bAbI HF-MoE] 400/1500 ce~1.8163 elapsed=431.2s
[bAbI HF-MoE] 500/1500 ce~1.8099 elapsed=539.3s
[bAbI HF-MoE] 600/1500 ce~1.8097 elapsed=647.3s
[bAbI HF-MoE] 700/1500 ce~1.8069 elapsed=755.4s
[bAbI HF-MoE] 800/1500 ce~1.8048 elapsed=863.6s
[bAbI HF-MoE] 900/1500 ce~1.7999 elapsed=971.6s
[bAbI HF-MoE] 1000/1500 ce~1.7662 elapsed=1079.6s
[bAbI HF-MoE] 1100/1500 ce~1.7203 elapsed=1187.6s
[bAbI HF-MoE] 1200/1500 ce~1.7101 elapsed=1295.6s
[bAbI HF-MoE] 1300/1500 ce~1.7138 elapsed=1403.6s
[bAbI HF-MoE] 1400/1500 ce~1.7162 elapsed=1511.6s
[bAbI HF-MoE] 1500/1500 ce~1.7093 elapsed=1619.6s
BABI HF-MoE RESULT {'seed': 11, 'seconds': 1619.802460193634, 'accuracy': 0.271, 'params': 7098384, 'steps': 1500}
[[ 22   0  98  29   0   0]
 [ 14   8 105  44   0   0]
 [  0   0 187   0   0   0]
 [  3   0  97  54   0   0]
 [  3   0  97  57

In [22]:
import hashlib, tempfile, os, shutil, gc

del babi_hf_moe
gc.collect(); torch.cuda.empty_cache()
assert USE_OXTA, 'este teste exige OxtaMem nativo'

def babi_entity(text, is_question=False):
    ww=words(text)
    return ww[2] if is_question else ww[0]

def babi_key_vector(example_id, entity):
    seed_bytes=hashlib.sha256(f'babi-qa1:{example_id}:{entity}'.encode()).digest()[:8]
    seed_int=int.from_bytes(seed_bytes,'little')
    rng=np.random.default_rng(seed_int)
    v=rng.standard_normal(BABI_DMODEL).astype(np.float32)
    v/=np.linalg.norm(v)+1e-8
    return v.tolist()

def sentence_ids(sent):
    return [babi_stoi.get(w,babi_stoi['<unk>']) for w in words(sent)]

root=tempfile.mkdtemp(prefix='oxta_babi_qa1_',dir='/content')
store=PyGeodesicEngine(os.path.join(root,'babi_store.db'),8)
store.set_sync_on_write(False)
expected_payload={}
for ei,ex in enumerate(babi_test_ex):
    latest={}
    for sent in ex['context']:
        latest[babi_entity(sent,False)]=sent
    for entity,sent in latest.items():
        payload=bytes(sentence_ids(sent))
        expected_payload[(ei,entity)]=payload
        store.write_with_vector(f'babi-{ei}-{entity}',payload,babi_key_vector(ei,entity))
store.flush()

retr_ok=0; model_ok=0; total=len(babi_test_ex)
for ei,ex in enumerate(babi_test_ex):
    entity=babi_entity(ex['question'],True)
    got=store.search_similar(babi_key_vector(ei,entity),1)
    payload=got[0] if got else b''
    retr_ok+=int(payload==expected_payload[(ei,entity)])
    qids=[babi_stoi.get(w,babi_stoi['<unk>']) for w in words(ex['question'])]
    prompt=list(payload)+[babi_stoi['<sep>']]+qids+[babi_stoi['<answer>']]
    prompt=left_pad(prompt)
    lg=np.asarray(babi_nsos.forward_ids(prompt).cpu().numpy()).reshape(-1,BABI_VOCAB)
    pred=int(lg[-1].argmax())
    true=babi_stoi[ex['answer'].lower()]
    model_ok+=int(pred==true)

babi_oxta_result={'retrieval_at_1':retr_ok/total,'accuracy':model_ok/total,
                  'model_only_accuracy':babi_nsos_acc,'test_examples':total,
                  'retrieval':'entity-keyed deterministic vectors; latest sentence per entity',
                  'uses_supporting_ids':False,'uses_test_answer_for_retrieval':False}
print('BABI NSOS+OXTAMEM RESULT',babi_oxta_result)
assert babi_oxta_result['retrieval_at_1']>=0.999
print('BABI_NSOS_OXTAMEM_PUBLIC_DONE')
del store; gc.collect(); shutil.rmtree(root,ignore_errors=True)

BABI NSOS+OXTAMEM RESULT {'retrieval_at_1': 1.0, 'accuracy': 1.0, 'model_only_accuracy': 0.767, 'test_examples': 1000, 'retrieval': 'entity-keyed deterministic vectors; latest sentence per entity', 'uses_supporting_ids': False, 'uses_test_answer_for_retrieval': False}
BABI_NSOS_OXTAMEM_PUBLIC_DONE


## 10 â€” Benchmark pÃºblico de linguagem natural: bAbI QA1

**Fonte pÃºblica:** facebook/babi_qa, configuraÃ§Ã£o en-10k-qa1, revisÃ£o imutÃ¡vel 1d86ad39d1c3ea2ff4b77eeb85f7c6ebd622a95f. Foram usados os Parquets oficiais do Hugging Face: 10.000 perguntas de treino e 1.000 de teste. LicenÃ§a CC BY 3.0.

**Integridade:** treino SHA-256 f1d67aa230d7aba0ed310df0d696a3ba9a07270e1670fe64c6901c24e5016f3f; teste SHA-256 9875dfad271cbfa5c49adb5809dd67be3826394a5d4d66dc74cab0c81483e2f8.

**PrÃ©-processamento:** tokenizaÃ§Ã£o por palavras e pontuaÃ§Ã£o; vocabulÃ¡rio construÃ­do apenas no treino (25 tokens); zero UNK no teste; comprimento mÃ¡ximo 83; baseline majoritÃ¡rio 18,7%. Todos os modelos receberam a histÃ³ria anterior completa e a pergunta, com padding Ã  esquerda.

**Rodada comparÃ¡vel:** semente 11, 1.500 passos, batch 32, LR 0,002, quatro camadas [Mamba, Attention, Mamba, Attention], dimensÃ£o 128 e estado Mamba 64.

| BraÃ§o | ParÃ¢metros | AcurÃ¡cia teste | Tempo T4 |
|---|---:|---:|---:|
| Jamba oficial denso | 1.196.048 | 49,3% | 1.590,6 s |
| Jamba oficial MoE 16/top-2 | 7.098.384 | 27,1% | 1.619,8 s |
| NSOS sem memÃ³ria | 609.304 | 76,7% | 93,7 s |
| NSOS + OxtaMem | mesmo modelo | 100,0% | avaliaÃ§Ã£o |

**OxtaMem:** retrieval@1 de 100,0% em 1.000 perguntas. A memÃ³ria foi chaveada por entidade extraÃ­da da pergunta e das sentenÃ§as; armazenou a sentenÃ§a completa mais recente da entidade; o modelo recebeu a sentenÃ§a recuperada mais a pergunta. NÃ£o foram usados supporting_ids nem respostas de teste para recuperar.

**Leitura honesta:** o NSOS superou o Jamba denso por 27,4 pontos e o MoE por 49,6 pontos nesta semente e orÃ§amento. O OxtaMem eliminou os distratores quando recebeu uma chave de entidade determinÃ­stica. Isso valida linguagem natural pÃºblica e memÃ³ria estruturada; ainda nÃ£o valida retrieval semÃ¢ntico livre, mÃºltiplas sementes neste dataset nem sucesso clÃ¡ssico acima de 95% do modelo isolado.

Os tempos do Jamba usam fallback PyTorch sem kernels Mamba fundidos e nÃ£o representam throughput de produÃ§Ã£o.


In [23]:
babi_nsos_seed_results=[{'seed':11,'data_seed':200011,'seconds':babi_nsos_s,
                         'accuracy':babi_nsos_acc,'params':sum(int(p.data.size) for p in babi_nsos.parameters())}]
for seed in (12,13):
    m=babi_build_nsos(seed)
    trn=nsos.Trainer(m,BABI_LR); trn.warmup_steps=100; trn.total_training_steps=BABI_STEPS
    trn.phase_scheduler.progressive_qat_enabled=False
    rr=random.Random(200000+seed); m.set_training_mode(True); t0=time.time(); run=0.0
    for step in range(BABI_STEPS):
        idx=[rr.randrange(len(babi_train_padded)) for _ in range(BABI_BATCH)]
        ps=[babi_train_padded[i][0] for i in idx]
        an=[[babi_train_padded[i][1]] for i in idx]
        run+=trn.train_supervised_batch(ps,an)
        if (step+1)%300==0:
            print(f'[bAbI NSOS seed={seed}] {step+1}/{BABI_STEPS} loss~{run/300:.4f}')
            run=0.0
    m.set_training_mode(False); elapsed=time.time()-t0
    acc,conf,_=babi_test_nsos(m)
    rec={'seed':seed,'data_seed':200000+seed,'seconds':elapsed,'accuracy':acc,
         'params':sum(int(p.data.size) for p in m.parameters())}
    babi_nsos_seed_results.append(rec); print('BABI NSOS SEED RESULT',rec)
    del m,trn; gc.collect()
    if hasattr(nsos,'release_cached_memory'): nsos.release_cached_memory()
vals=[r['accuracy'] for r in babi_nsos_seed_results]
babi_nsos_multiseed={'mean':float(np.mean(vals)),'std_sample':float(np.std(vals,ddof=1)),
                      'runs':vals,'seeds':[11,12,13]}
print('BABI NSOS MULTISEED',babi_nsos_multiseed)
print('BABI_NSOS_3_SEEDS_DONE')

[bAbI NSOS seed=12] 300/1500 loss~3.6787
[bAbI NSOS seed=12] 600/1500 loss~2.6849
[bAbI NSOS seed=12] 900/1500 loss~2.5761
[bAbI NSOS seed=12] 1200/1500 loss~2.4171
[bAbI NSOS seed=12] 1500/1500 loss~2.2232
BABI NSOS SEED RESULT {'seed': 12, 'data_seed': 200012, 'seconds': 93.83603429794312, 'accuracy': 0.652, 'params': 609304}
[bAbI NSOS seed=13] 300/1500 loss~3.6526
[bAbI NSOS seed=13] 600/1500 loss~2.8332
[bAbI NSOS seed=13] 900/1500 loss~2.6746
[bAbI NSOS seed=13] 1200/1500 loss~2.5788
[bAbI NSOS seed=13] 1500/1500 loss~2.2882
BABI NSOS SEED RESULT {'seed': 13, 'data_seed': 200013, 'seconds': 93.33760452270508, 'accuracy': 0.637, 'params': 609304}
BABI NSOS MULTISEED {'mean': 0.6853333333333333, 'std_sample': 0.07112196097783956, 'runs': [0.767, 0.652, 0.637], 'seeds': [11, 12, 13]}
BABI_NSOS_3_SEEDS_DONE


### 10.1 â€” Sementes de treino adicionais do NSOS no bAbI

O NSOS foi repetido com sementes 11, 12 e 13 no mesmo protocolo pÃºblico. AcurÃ¡cias: 76,7%, 65,2% e 63,7%; mÃ©dia 68,5% Â± 7,1% (desvio-padrÃ£o amostral). Todas ficaram acima do baseline majoritÃ¡rio de 18,7%.

A semente 11 foi a melhor, portanto 76,7% nÃ£o representa o desempenho esperado isoladamente. O comparativo Jamba denso/MoE ainda possui apenas a semente 11 neste dataset, devido ao custo do fallback PyTorch (aproximadamente 27 minutos por treino). NÃ£o calcular significÃ¢ncia entre arquiteturas no bAbI atÃ© repetir os rivais.


In [24]:
SCALE_SEED=21
SCALE_EVAL_TOTAL=400
SCALE_PAIRS=(8,16,32,64)
scale_eval_sets={}
for n in SCALE_PAIRS:
    rr=random.Random(400000+n)
    pairs=[mqar_seq(rr,n) for _ in range(SCALE_EVAL_TOTAL)]
    scale_eval_sets[n]=([x[0] for x in pairs],[x[1] for x in pairs])

def scale_nsos_acc(m,n):
    seqs,ys=scale_eval_sets[n]; ok=0
    for seq,y in zip(seqs,ys):
        lg=np.asarray(m.forward_ids(seq).cpu().numpy()).reshape(-1,VOCAB)
        ok+=int(int(lg[-1].argmax())==y)
    return ok/len(ys)

@torch.no_grad()
def scale_hf_acc(m,n,chunk=64):
    seqs,ys=scale_eval_sets[n]; ps=[]
    for st in range(0,len(seqs),chunk):
        ids=torch.tensor(seqs[st:st+chunk],device='cuda',dtype=torch.long)
        ps.extend(m(ids,use_cache=False).logits[:,-1,:].argmax(-1).cpu().tolist())
    return sum(a==b for a,b in zip(ps,ys))/len(ys)

# NSOS caminho GPU rapido, semente nao nula.
nsos.set_seed(SCALE_SEED)
if hasattr(nsos,'set_deterministic_reductions'): nsos.set_deterministic_reductions(False)
scale_nsos=build(); scale_tr=nsos.Trainer(scale_nsos,LR)
scale_tr.warmup_steps=100; scale_tr.total_training_steps=STEPS
scale_tr.phase_scheduler.progressive_qat_enabled=False
rr=random.Random(300000+SCALE_SEED); scale_nsos.set_training_mode(True); t0=time.time(); run=0.0
for step in range(STEPS):
    npair=rr.randint(1,N_PAIRS); ps,an=batch(rr,npair)
    run+=scale_tr.train_supervised_batch(ps,an)
    if (step+1)%500==0:
        print(f'[SCALE NSOS] {step+1}/{STEPS} loss~{run/500:.3f}'); run=0.0
scale_nsos.set_training_mode(False); scale_nsos_s=time.time()-t0
print('SCALE NSOS trained',scale_nsos_s)

def train_scale_hf(cfg,label,use_moe):
    torch.manual_seed(SCALE_SEED); torch.cuda.manual_seed_all(SCALE_SEED)
    m=JambaForCausalLM(cfg).cuda().float()
    opt=torch.optim.AdamW(m.parameters(),lr=LR,betas=(0.9,0.999),eps=1e-8,weight_decay=0.01)
    rr=random.Random(300000+SCALE_SEED); m.train(); t0=time.time(); run=0.0
    for step in range(STEPS):
        npair=rr.randint(1,N_PAIRS); seqs=[]; ys=[]
        for _ in range(BATCH):
            seq,y=mqar_seq(rr,npair); seqs.append(seq); ys.append(y)
        ids=torch.tensor(seqs,device='cuda',dtype=torch.long)
        target=torch.tensor(ys,device='cuda',dtype=torch.long)
        for group in opt.param_groups: group['lr']=ms_lr_at(step)
        out=m(ids,use_cache=False,output_router_logits=use_moe)
        ce=F.cross_entropy(out.logits[:,-1,:],target)
        loss=ce+(cfg.router_aux_loss_coef*out.aux_loss if use_moe else 0.0)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); run+=float(ce.detach())
        if (step+1)%500==0:
            print(f'[SCALE {label}] {step+1}/{STEPS} ce~{run/500:.3f}'); run=0.0
    elapsed=time.time()-t0; m.eval(); return m,elapsed

scale_dense,scale_dense_s=train_scale_hf(hf_cfg,'HF-dense',False)
scale_moe,scale_moe_s=train_scale_hf(hf_moe_cfg_ms,'HF-MoE',True)
scale_extrapolation=[]
print('MQAR EXTRAPOLATION 1-8 train -> 8/16/32/64 eval')
for n in SCALE_PAIRS:
    row={'pairs':n,'tokens':2*n+2,'nsos':scale_nsos_acc(scale_nsos,n),
         'hf_dense':scale_hf_acc(scale_dense,n),'hf_moe':scale_hf_acc(scale_moe,n)}
    scale_extrapolation.append(row); print(row)
scale_artifact={'seed':SCALE_SEED,'train_pairs':[1,8],'train_steps':STEPS,
                'eval_examples_each':SCALE_EVAL_TOTAL,'results':scale_extrapolation,
                'times':{'nsos':scale_nsos_s,'hf_dense':scale_dense_s,'hf_moe':scale_moe_s}}
print('SCALE_ARTIFACT',scale_artifact)
print('MQAR_SCALE_EXTRAPOLATION_DONE')

[SCALE NSOS] 500/2500 loss~8.343
[SCALE NSOS] 1000/2500 loss~2.005
[SCALE NSOS] 1500/2500 loss~0.389
[SCALE NSOS] 2000/2500 loss~0.172
[SCALE NSOS] 2500/2500 loss~0.112
SCALE NSOS trained 34.626384973526
[SCALE HF-dense] 500/2500 ce~3.582
[SCALE HF-dense] 1000/2500 ce~2.643
[SCALE HF-dense] 1500/2500 ce~2.367
[SCALE HF-dense] 2000/2500 ce~2.098
[SCALE HF-dense] 2500/2500 ce~2.075
[SCALE HF-MoE] 500/2500 ce~3.536
[SCALE HF-MoE] 1000/2500 ce~2.588
[SCALE HF-MoE] 1500/2500 ce~2.219
[SCALE HF-MoE] 2000/2500 ce~1.970
[SCALE HF-MoE] 2500/2500 ce~1.976
MQAR EXTRAPOLATION 1-8 train -> 8/16/32/64 eval
{'pairs': 8, 'tokens': 18, 'nsos': 0.9575, 'hf_dense': 0.125, 'hf_moe': 0.1275}
{'pairs': 16, 'tokens': 34, 'nsos': 0.7825, 'hf_dense': 0.075, 'hf_moe': 0.0625}
{'pairs': 32, 'tokens': 66, 'nsos': 0.4925, 'hf_dense': 0.0425, 'hf_moe': 0.0475}
{'pairs': 64, 'tokens': 130, 'nsos': 0.285, 'hf_dense': 0.0225, 'hf_moe': 0.0275}
SCALE_ARTIFACT {'seed': 21, 'train_pairs': [1, 8], 'train_steps': 2500, 'ev

## 11 â€” Escala e extrapolaÃ§Ã£o de comprimento MQAR

Rodada independente com semente 21 nÃ£o nula. Todos os braÃ§os treinaram por 2.500 passos apenas com 1â€“8 pares e foram testados em 400 exemplos fixos por comprimento, atÃ© 64 pares (130 tokens).

| Pares | Tokens | Jamba denso | Jamba MoE | NSOS |
|---:|---:|---:|---:|---:|
| 8 | 18 | 12,5% | 12,8% | 95,8% |
| 16 | 34 | 7,5% | 6,3% | 78,3% |
| 32 | 66 | 4,3% | 4,8% | 49,3% |
| 64 | 130 | 2,3% | 2,8% | 28,5% |

O NSOS perde acurÃ¡cia Ã  medida que o comprimento extrapola 2x, 4x e 8x o mÃ¡ximo de treino, mas mantÃ©m vantagem absoluta grande em todos os pontos. Em 64 pares, ainda fica cerca de 10 vezes acima dos rivais e muito acima do acaso de 1,56%. O resultado usa uma Ãºnica semente de treino e deve ser classificado como evidÃªncia preliminar de extrapolaÃ§Ã£o, nÃ£o como curva estatÃ­stica final.
